# 🖼️ Image Captioning — CNN + RNN (from scratch)

**Goal:** Given a waste image → generate a text description.

**Architecture:**
```
Image → CNN (feature extractor) → feature vector
                                        ↓
                               RNN Decoder (word by word)
                                        ↓
                               Generated Caption
```

**Pipeline:**
1. Install & Import
2. Load & Prepare Data
3. Tokenize Captions
4. Build CNN Encoder
5. Build RNN Decoder
6. Build Full Captioning Model
7. Train
8. Generate Caption for New Image
9. Save

## 0. Install & Import

In [ ]:
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['tensorflow', 'pandas', 'numpy', 'matplotlib', 'Pillow', 'tqdm']:
    install(pkg)

print('✅ All packages ready.')

In [ ]:
import os, re, string, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, GlobalAveragePooling2D,
    Embedding, SimpleRNN, Dense, Dropout, Add
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print(f'✅ TensorFlow {tf.__version__}')

## 1. Config — Edit These Paths

In [ ]:
# ── Paths ──────────────────────────────────────────────────
CSV_PATH   = 'processed/train.csv'   # output from preprocessing notebook
VAL_CSV    = 'processed/val.csv'

TEXT_COL   = 'text_clean'            # cleaned caption column
IMAGE_COL  = 'image_path'            # relative or absolute image paths

# ── Image settings ─────────────────────────────────────────
IMG_H, IMG_W = 64, 64                # resize all images to this
IMG_CHANNELS = 3

# ── Text settings ──────────────────────────────────────────
VOCAB_SIZE   = 5_000
MAX_LEN      = 30                    # max caption length (words)

# ── Model settings ─────────────────────────────────────────
EMBED_DIM    = 128
RNN_UNITS    = 256
DENSE_UNITS  = 256
DROPOUT      = 0.4

# ── Training ───────────────────────────────────────────────
EPOCHS       = 30
BATCH_SIZE   = 32
SEED         = 42

print('✅ Config loaded.')

## 2. Load Data & Prepare Captions

In [ ]:
train_df = pd.read_csv(CSV_PATH)
val_df   = pd.read_csv(VAL_CSV)

# ── Add <start> and <end> tokens to every caption ──
# The RNN needs to know when to start and stop generating.
def add_tokens(text):
    return '<start> ' + str(text).strip() + ' <end>'

train_df['caption'] = train_df[TEXT_COL].apply(add_tokens)
val_df['caption']   = val_df[TEXT_COL].apply(add_tokens)

print(f'Train samples : {len(train_df):,}')
print(f'Val   samples : {len(val_df):,}')
print(f'Sample caption: {train_df["caption"].iloc[0]}')

## 3. Tokenize Captions

In [ ]:
tokenizer = Tokenizer(
    num_words = VOCAB_SIZE,
    oov_token = '<OOV>',
    filters   = '"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n'  # keep ! and '
)
tokenizer.fit_on_texts(train_df['caption'].tolist())

VOCAB_SIZE_ACTUAL = min(VOCAB_SIZE, len(tokenizer.word_index) + 1)

print(f'Vocab size (actual) : {len(tokenizer.word_index):,}')
print(f'Vocab size (used)   : {VOCAB_SIZE_ACTUAL:,}')

# ── Helper: index of special tokens ──
START_IDX = tokenizer.word_index.get('<start>', 1)
END_IDX   = tokenizer.word_index.get('<end>',   2)
print(f'<start> index: {START_IDX}  |  <end> index: {END_IDX}')

## 4. Image Loading Utility

In [ ]:
def load_image(path: str) -> np.ndarray:
    """Load, resize, and normalise a single image → (H, W, 3) float32 in [0,1]."""
    img = Image.open(path).convert('RGB').resize((IMG_W, IMG_H))
    return np.array(img, dtype=np.float32) / 255.0


# ── Quick sanity check ──
sample_path = train_df[IMAGE_COL].iloc[0]
sample_img  = load_image(sample_path)
print(f'Image shape : {sample_img.shape}')    # should be (64, 64, 3)
print(f'Pixel range : [{sample_img.min():.2f}, {sample_img.max():.2f}]')

plt.imshow(sample_img)
plt.title(train_df['caption'].iloc[0])
plt.axis('off')
plt.show()

## 5. Data Generator

Each training step gets **(image, partial_caption) → next_word**.

Example for caption `<start> plastic bottle <end>`:
```
image + [<start>]              → plastic
image + [<start>, plastic]     → bottle
image + [<start>, plastic, bottle] → <end>
```

In [ ]:
def make_dataset(df: pd.DataFrame, batch_size: int, shuffle: bool = True):
    """
    Build a tf.data.Dataset that yields:
        ({image: (B,H,W,3), partial_seq: (B, MAX_LEN)}, next_word_onehot: (B, VOCAB))
    """
    images_list, seqs_list, targets_list = [], [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc='Building dataset'):
        try:
            img     = load_image(row[IMAGE_COL])                        # (H, W, 3)
            seq     = tokenizer.texts_to_sequences([row['caption']])[0] # list of ints

            # Generate one sample per step in the caption
            for i in range(1, len(seq)):
                partial = pad_sequences([seq[:i]], maxlen=MAX_LEN, padding='post')[0]
                target  = to_categorical([seq[i]], num_classes=VOCAB_SIZE_ACTUAL)[0]

                images_list.append(img)
                seqs_list.append(partial)
                targets_list.append(target)
        except Exception:
            pass   # skip broken image paths

    images_arr  = np.array(images_list,  dtype=np.float32)
    seqs_arr    = np.array(seqs_list,    dtype=np.int32)
    targets_arr = np.array(targets_list, dtype=np.float32)

    dataset = tf.data.Dataset.from_tensor_slices(
        ({'image': images_arr, 'partial_seq': seqs_arr}, targets_arr)
    )
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(images_list), seed=SEED)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)


print('Building training dataset …')
train_ds = make_dataset(train_df, BATCH_SIZE, shuffle=True)
print('Building validation dataset …')
val_ds   = make_dataset(val_df,   BATCH_SIZE, shuffle=False)

print(f'\n✅ Datasets ready.')
# Peek at one batch
for (batch_x, batch_y) in train_ds.take(1):
    print(f"  image shape      : {batch_x['image'].shape}")
    print(f"  partial_seq shape: {batch_x['partial_seq'].shape}")
    print(f"  target shape     : {batch_y.shape}")

## 6. Build the Model

```
Image Branch:       image (64×64×3)
                       ↓
                    CNN (3 conv blocks)
                       ↓
                    GlobalAvgPool → Dense(DENSE_UNITS)

Caption Branch:     partial_seq (MAX_LEN,)
                       ↓
                    Embedding → SimpleRNN → Dense(DENSE_UNITS)

Merge:              Add(image_feat, rnn_feat)
                       ↓
                    Dense(VOCAB_SIZE, softmax)  → next word
```

In [ ]:
def build_captioning_model(img_h, img_w, vocab_size, embed_dim,
                            rnn_units, dense_units, max_len, dropout):

    # ── Image branch (CNN Encoder) ──────────────────────────
    img_input = Input(shape=(img_h, img_w, IMG_CHANNELS), name='image')

    x = Conv2D(32, (3,3), activation='relu', padding='same')(img_input)
    x = MaxPooling2D()(x)

    x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = MaxPooling2D()(x)

    x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = GlobalAveragePooling2D()(x)

    img_feat = Dense(dense_units, activation='relu', name='img_feat')(x)
    img_feat = Dropout(dropout)(img_feat)

    # ── Caption branch (RNN Decoder) ────────────────────────
    seq_input = Input(shape=(max_len,), name='partial_seq')

    emb = Embedding(vocab_size, embed_dim, mask_zero=True)(seq_input)
    rnn_out = SimpleRNN(rnn_units, name='rnn')(emb)
    seq_feat = Dense(dense_units, activation='relu', name='seq_feat')(rnn_out)
    seq_feat = Dropout(dropout)(seq_feat)

    # ── Merge & predict next word ────────────────────────────
    merged = Add()([img_feat, seq_feat])
    output = Dense(vocab_size, activation='softmax', name='output')(merged)

    model = Model(
        inputs  = [img_input, seq_input],
        outputs = output,
        name    = 'WasteCaptioner'
    )
    return model


model = build_captioning_model(
    img_h       = IMG_H,
    img_w       = IMG_W,
    vocab_size  = VOCAB_SIZE_ACTUAL,
    embed_dim   = EMBED_DIM,
    rnn_units   = RNN_UNITS,
    dense_units = DENSE_UNITS,
    max_len     = MAX_LEN,
    dropout     = DROPOUT
)

model.summary()

## 7. Compile & Train

In [ ]:
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_captioner.keras', monitor='val_loss',
                    save_best_only=True, verbose=1)
]

history = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = EPOCHS,
    callbacks       = callbacks
)

## 8. Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric in zip(axes, ['accuracy', 'loss']):
    ax.plot(history.history[metric],          label=f'Train {metric}', color='steelblue')
    ax.plot(history.history[f'val_{metric}'], label=f'Val   {metric}', color='orange')
    ax.set_title(metric.capitalize(), fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

## 9. Generate Caption for a New Image

The model generates the caption **word by word** (greedy decoding):
1. Start with `<start>`
2. Feed image + partial sequence → predict next word
3. Append predicted word → repeat until `<end>` or MAX_LEN

In [ ]:
# ── Reverse lookup: index → word ──
idx2word = {v: k for k, v in tokenizer.word_index.items()}


def generate_caption(image_path: str, max_len: int = MAX_LEN) -> str:
    """
    Given a path to a waste image, return a generated text description.
    """
    img   = load_image(image_path)                    # (H, W, 3)
    img   = np.expand_dims(img, axis=0)               # (1, H, W, 3)

    # Start with <start> token
    seq   = [START_IDX]

    for _ in range(max_len):
        padded  = pad_sequences([seq], maxlen=max_len, padding='post')
        preds   = model.predict({'image': img, 'partial_seq': padded}, verbose=0)
        next_id = int(np.argmax(preds[0]))            # greedy: pick highest prob word

        if next_id == END_IDX:
            break

        seq.append(next_id)

    # Convert indices back to words (skip <start>)
    words = [idx2word.get(i, '') for i in seq[1:]]
    return ' '.join(w for w in words if w not in ('', '<OOV>'))


# ── Test on a few samples ──
print('Generated captions on validation samples:\n')
for _, row in val_df.head(5).iterrows():
    generated = generate_caption(row[IMAGE_COL])
    real      = row[TEXT_COL]
    print(f'  Real      : {real}')
    print(f'  Generated : {generated}')
    print('  ' + '-'*50)

    # Show the image
    img_show = Image.open(row[IMAGE_COL]).resize((150, 150))
    plt.imshow(img_show)
    plt.title(f'Generated: {generated}', fontsize=9)
    plt.axis('off')
    plt.show()

## 10. Predict on a Brand-New Image

In [ ]:
# ── Change this path to any image you want ──
NEW_IMAGE_PATH = 'your_image.jpg'

caption = generate_caption(NEW_IMAGE_PATH)
print(f'Caption: {caption}')

img_show = Image.open(NEW_IMAGE_PATH).resize((200, 200))
plt.imshow(img_show)
plt.title(caption, fontsize=10, wrap=True)
plt.axis('off')
plt.show()

## 11. Save Everything

In [ ]:
os.makedirs('saved_captioner', exist_ok=True)

model.save('saved_captioner/captioning_model.keras')

with open('saved_captioner/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# Save config so you can reload without hardcoding
config = {
    'VOCAB_SIZE_ACTUAL': VOCAB_SIZE_ACTUAL,
    'MAX_LEN'          : MAX_LEN,
    'IMG_H'            : IMG_H,
    'IMG_W'            : IMG_W,
    'START_IDX'        : START_IDX,
    'END_IDX'          : END_IDX
}
with open('saved_captioner/config.pkl', 'wb') as f:
    pickle.dump(config, f)

print('✅ Saved to ./saved_captioner/')
print('   ├── captioning_model.keras')
print('   ├── tokenizer.pkl')
print('   └── config.pkl')